# v3.0.0 PD PPG-AST -- Phonetic Feature Experiment

Compare foundation-model Posterior Phoneme Gram (PPG) features vs raw spectrograms
for PD screening on Bridge2AI v3.0.0 (833 participants).

Pipeline is IDENTICAL to the spectrogram experiment (`v3_PD_AST_Spectrograms.ipynb`)
except the input is PPGs instead of mel spectrograms:
- PPG data: `features/ppgs.parquet` (column: `ppgs`)
- PPG shape per recording: (40 phonemes, T time frames) stored as float16
- Processing: pad/crop temporal dimension to 1024, then resize (40, 1024) -> (128, 1024)
- Same z-score normalization, same ASTClassifier, same FocalLoss, same CV splits

Same 5-fold StratifiedKFold (random_state=42) ensures results are directly comparable.

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

ROOT = Path('/data0/b2ai-voice/3.0.0')
PPG_PATH = ROOT / 'features' / 'ppgs.parquet'
PD_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'parkinsons_disease.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Path to spectrogram results for final comparison
SPEC_RESULTS = RESULTS_DIR / 'ast_pd_v3_cv_results.npz'

print(f'PPG exists: {PPG_PATH.exists()}')
print(f'PD phen exists: {PD_PHEN.exists()}')
print(f'Ctrl phen exists: {CTRL_PHEN.exists()}')
print(f'Spec results exist: {SPEC_RESULTS.exists()}')

In [ ]:
#2 - Load PPG data (row-group reads for memory efficiency)
pf = pq.ParquetFile(PPG_PATH)
parts = []
for i in range(pf.num_row_groups):
    rg = pf.read_row_group(i, columns=['participant_id', 'session_id', 'task_name', 'ppgs']).to_pandas()
    parts.append(rg)
ppg_df = pd.concat(parts, ignore_index=True)

# Zero-pad participant IDs to 6 digits
ppg_df['participant_id'] = ppg_df['participant_id'].astype(str).str.zfill(6)

# Derive n_frames from the PPG arrays (axis 1 = time)
ppg_df['n_frames'] = ppg_df['ppgs'].apply(lambda x: np.asarray(x).shape[1] if hasattr(np.asarray(x), 'shape') and np.asarray(x).ndim == 2 else 0)

print(f'Total recordings: {len(ppg_df)}')
print(f'Unique participants: {ppg_df["participant_id"].nunique()}')
print(f'Unique tasks: {ppg_df["task_name"].nunique()}')

# Inspect a sample PPG shape
sample = np.asarray(ppg_df['ppgs'].iloc[0])
print(f'Sample PPG shape: {sample.shape}, dtype: {sample.dtype}')

In [ ]:
#3 - Build PD labels from hierarchical phenotype
pd_df = pd.read_csv(PD_PHEN, sep='\t')
ctrl_df = pd.read_csv(CTRL_PHEN, sep='\t')

pd_ids = set(pd_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_df['participant_id'].astype(str).str.zfill(6))

# Remove any PD-Control overlap
overlap = pd_ids & ctrl_ids
ctrl_ids_clean = ctrl_ids - overlap
print(f'PD participants: {len(pd_ids)}')
print(f'Control participants (clean): {len(ctrl_ids_clean)}')
print(f'Overlap removed: {len(overlap)}')

# Assign labels
ppg_df['label'] = np.nan
ppg_df.loc[ppg_df['participant_id'].isin(pd_ids), 'label'] = 1
ppg_df.loc[ppg_df['participant_id'].isin(ctrl_ids_clean), 'label'] = 0

# Keep only labeled recordings
data = ppg_df.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
print(f'\nLabeled recordings: {len(data)}')
print(f'PD recordings: {(data["label"]==1).sum()}')
print(f'Control recordings: {(data["label"]==0).sum()}')
print(f'Unique participants: {data["participant_id"].nunique()}')

In [ ]:
#4 - Task selection: same 8 tasks as spectrogram experiments
SELECTED_TASKS = [
    'prolonged-vowel',           # sustained phonation
    'glides-high-to-low',        # pitch control
    'glides-low-to-high',        # pitch control
    'diadochokinesis-pataka',    # articulatory agility
    'rainbow-passage',           # connected speech
    'picture-description',       # cognitive-linguistic
    'story-recall',              # memory + speech
    'maximum-phonation-time-1',  # sustained phonation
]

MIN_TIME_FRAMES = 100

data_selected = data[
    (data['task_name'].isin(SELECTED_TASKS)) &
    (data['n_frames'] >= MIN_TIME_FRAMES)
].copy()

print(f'Selected task recordings: {len(data_selected)}')
print(f'PD: {(data_selected["label"]==1).sum()} recordings from {data_selected[data_selected["label"]==1]["participant_id"].nunique()} participants')
print(f'Ctrl: {(data_selected["label"]==0).sum()} recordings from {data_selected[data_selected["label"]==0]["participant_id"].nunique()} participants')
print(f'Total participants: {data_selected["participant_id"].nunique()}')
print(f'\nPer-task counts:')
print(data_selected.groupby('task_name')['label'].value_counts().unstack(fill_value=0))

In [ ]:
#5 - Process PPGs: pad/crop to 1024 time frames
from tqdm import tqdm

TARGET_SEQ_LEN = 1024

def process_ppg(ppg_raw, target_len=1024):
    """Process raw PPG with reflect padding/center crop on temporal dimension.

    Input shape: (40 phonemes, T time frames) as float16
    Output shape: (40, target_len) as float32
    """
    ppg = np.asarray(ppg_raw).astype(np.float32)
    n_phones, time_len = ppg.shape
    if time_len < target_len:
        pad_width = target_len - time_len
        ppg = np.pad(ppg, ((0, 0), (0, pad_width)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        ppg = ppg[:, start:start + target_len]
    return ppg

X_list = []
for idx, row in tqdm(data_selected.iterrows(), total=len(data_selected), desc='Processing PPGs'):
    processed = process_ppg(row['ppgs'], TARGET_SEQ_LEN)
    X_list.append(processed)

X_raw = np.stack(X_list)
y_raw = data_selected['label'].values
participants_raw = data_selected['participant_id'].values
tasks_raw = data_selected['task_name'].values

print(f'Processed shape: {X_raw.shape}')  # (N, 40, 1024)
print(f'Value range: [{X_raw.min():.4f}, {X_raw.max():.4f}]')

In [ ]:
#6 - AST model definition and helpers (identical to spectrogram pipeline)
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig
from scipy.ndimage import zoom

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def resize_ppg(ppg, target_freq=128, target_time=1024):
    """Resize PPG to AST expected dimensions (40 -> 128 frequency bins)."""
    current_freq, current_time = ppg.shape
    freq_ratio = target_freq / current_freq
    time_ratio = target_time / current_time
    resized = zoom(ppg, (freq_ratio, time_ratio), order=1)
    return resized.astype(np.float32)

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(hidden_size=768, num_hidden_layers=12,
                             num_attention_heads=12, intermediate_size=3072,
                             max_length=1024, num_mel_bins=128)
            self.ast = ASTModel(config)
            hidden_size = 768
        if freeze_base:
            for param in self.ast.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 128, 1024) -> (B, 1024, 128)
        outputs = self.ast(input_values=x)
        pooled = outputs.pooler_output
        return self.classifier(pooled)

class ASTDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            # SpecAugment: time masking (up to 150 frames, 50% probability)
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            # SpecAugment: frequency masking (up to 30 bins, 50% probability)
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def evaluate_fold(model, loader, device):
    """Evaluate model on a fold, returning participant-level aggregated predictions."""
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_parts = np.array(all_parts)
    # Participant-level aggregation (mean of recording-level probabilities)
    unique_parts = np.unique(all_parts)
    part_probs, part_labels = [], []
    for p in unique_parts:
        mask = all_parts == p
        part_probs.append(all_probs[mask].mean())
        part_labels.append(all_labels[mask][0])
    return np.array(part_probs), np.array(part_labels), unique_parts

print('Model and helpers defined.')

In [ ]:
#7 - 5-Fold Cross-Validation (same splits as spectrogram experiment)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import copy
import time

unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])

print(f'Total participants: {len(unique_participants)}')
print(f'PD: {int(participant_labels.sum())} ({participant_labels.mean():.1%})')
print(f'Control: {int((participant_labels == 0).sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []
all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
all_oof_labels = participant_labels.astype(np.int64).copy()
norm_stats = {}

total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f'\n{"="*60}')
    print(f'--- Fold {fold+1}/{N_FOLDS} ---')
    print(f'{"="*60}')

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    X_train_fold = X_raw[train_mask]
    y_train_fold = y_raw[train_mask]
    parts_train_fold = participants_raw[train_mask]

    X_val_fold = X_raw[val_mask]
    y_val_fold = y_raw[val_mask]
    parts_val_fold = participants_raw[val_mask]

    print(f'Train: {len(X_train_fold)} recordings from {len(train_parts_fold)} participants')
    print(f'Val: {len(X_val_fold)} recordings from {len(val_parts_fold)} participants')

    # Resize 40 -> 128 frequency bins for AST input
    X_train_ast = np.stack([resize_ppg(x) for x in tqdm(X_train_fold, desc='resize train', leave=False)])
    X_val_ast = np.stack([resize_ppg(x) for x in tqdm(X_val_fold, desc='resize val', leave=False)])

    # Fold-specific z-score normalization
    fold_mean = X_train_ast.mean()
    fold_std = X_train_ast.std()
    X_train_ast = (X_train_ast - fold_mean) / (fold_std + 1e-8)
    X_val_ast = (X_val_ast - fold_mean) / (fold_std + 1e-8)
    norm_stats[fold + 1] = {'mean': float(fold_mean), 'std': float(fold_std)}

    # Datasets
    train_ds = ASTDataset(X_train_ast, y_train_fold, parts_train_fold, augment=True)
    val_ds = ASTDataset(X_val_ast, y_val_fold, parts_val_fold, augment=False)

    # Balanced sampler
    class_counts = np.bincount(y_train_fold)
    weights = 1.0 / class_counts
    sample_weights = weights[y_train_fold]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    # Fresh model
    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)

    backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
    head_params = [p for n, p in model.named_parameters() if 'classifier' in n]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 5e-6, 'weight_decay': 0.01},
        {'params': head_params, 'lr': 5e-4, 'weight_decay': 0.01}
    ], betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

    # Dynamic class weights for FocalLoss
    cc = np.bincount(y_train_fold)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    class_weights_fold = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=class_weights_fold, gamma=2.0)
    print(f'Class weights: {cw}')

    best_score = 0
    best_state = None
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(30):
        model.train()
        total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Evaluate
        part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
        if len(np.unique(part_labels_fold)) > 1:
            auc = roc_auc_score(part_labels_fold, part_probs)
            fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
            opt_idx = np.argmax(tpr - fpr)
            opt_thresh = thresholds[opt_idx]
            preds_opt = (part_probs >= opt_thresh).astype(int)
            f1_opt = f1_score(part_labels_fold, preds_opt, zero_division=0)
        else:
            auc, f1_opt = 0.5, 0.0

        score = 0.4 * auc + 0.6 * f1_opt
        if score > best_score + 0.01:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '<-- best'
        else:
            patience_counter += 1
            marker = ''

        avg_loss = total_loss / len(train_loader)
        print(f'  Epoch {epoch+1:02d} | loss {avg_loss:.4f} | AUC {auc:.3f} | F1opt {f1_opt:.3f} {marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best and get final predictions
    model.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)

    # Save fold model
    torch.save(model.state_dict(), str(RESULTS_DIR / f'ppg_ast_pd_v3_fold{fold+1}.pt'))

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx_oof = np.where(unique_participants == pid)[0][0]
        all_oof_probs[idx_oof] = part_probs[i]

    # Fold metrics
    fold_auc = roc_auc_score(part_labels_fold, part_probs)
    fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
    opt_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[opt_idx]
    preds_opt = (part_probs >= opt_thresh).astype(int)

    fold_results.append({
        'fold': fold + 1,
        'auc': float(fold_auc),
        'f1_opt': float(f1_score(part_labels_fold, preds_opt, zero_division=0)),
        'recall_opt': float(recall_score(part_labels_fold, preds_opt, zero_division=0)),
        'precision_opt': float(precision_score(part_labels_fold, preds_opt, zero_division=0)),
        'threshold': float(opt_thresh)
    })

    print(f'Fold {fold+1}: AUC={fold_auc:.4f}, F1={fold_results[-1]["f1_opt"]:.4f}')

    del model, optimizer, train_ds, val_ds
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f'\nTotal CV time: {total_time/60:.1f} minutes')

In [ ]:
#8 - CV Summary and OOF evaluation
from scipy import stats
from sklearn.metrics import confusion_matrix, accuracy_score

aucs = [r['auc'] for r in fold_results]
f1s = [r['f1_opt'] for r in fold_results]
recalls = [r['recall_opt'] for r in fold_results]
precisions = [r['precision_opt'] for r in fold_results]

n = N_FOLDS
t_crit = stats.t.ppf(0.975, df=n - 1)

print('Per-fold results:')
for r in fold_results:
    print(f'  Fold {r["fold"]}: AUC={r["auc"]:.4f}  F1={r["f1_opt"]:.4f}  '
          f'Rec={r["recall_opt"]:.4f}  Prec={r["precision_opt"]:.4f}  Thr={r["threshold"]:.3f}')

print('\nMean +/- SD [95% CI]:')
for name, arr in [('AUC', aucs), ('F1', f1s), ('Recall', recalls), ('Precision', precisions)]:
    m = np.mean(arr)
    sd = np.std(arr, ddof=1)
    se = sd / np.sqrt(n)
    ci_lo = m - t_crit * se
    ci_hi = m + t_crit * se
    print(f'  {name}: {m:.4f} ({sd:.4f}) [{ci_lo:.4f}, {ci_hi:.4f}]')

# OOF metrics
oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr, tpr, thresholds = roc_curve(all_oof_labels, all_oof_probs)
opt_idx = np.argmax(tpr - fpr)
oof_thresh = thresholds[opt_idx]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)

oof_f1 = f1_score(all_oof_labels, oof_preds, zero_division=0)
oof_rec = recall_score(all_oof_labels, oof_preds, zero_division=0)
oof_prec = precision_score(all_oof_labels, oof_preds, zero_division=0)
oof_acc = accuracy_score(all_oof_labels, oof_preds)
cm = confusion_matrix(all_oof_labels, oof_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print(f'\nOOF AUC:         {oof_auc:.4f}')
print(f'OOF F1:          {oof_f1:.4f}')
print(f'OOF Recall:      {oof_rec:.4f}')
print(f'OOF Precision:   {oof_prec:.4f}')
print(f'OOF Specificity: {specificity:.4f}')
print(f'OOF Accuracy:    {oof_acc:.4f}')
print(f'Threshold:       {oof_thresh:.3f}')
print(f'\nConfusion Matrix: TP={tp} FP={fp} FN={fn} TN={tn}')

# Save results
np.savez(str(RESULTS_DIR / 'ppg_ast_pd_v3_cv_results.npz'),
    oof_probs=all_oof_probs,
    oof_labels=all_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array(aucs),
    fold_f1s=np.array(f1s),
    fold_recalls=np.array(recalls),
    fold_precisions=np.array(precisions),
    fold_thresholds=np.array([r['threshold'] for r in fold_results]),
    oof_auc=np.array(oof_auc),
    oof_thresh=np.array(oof_thresh),
    norm_means=np.array([norm_stats[f+1]['mean'] for f in range(N_FOLDS)]),
    norm_stds=np.array([norm_stats[f+1]['std'] for f in range(N_FOLDS)]),
)
print(f'\nSaved: {RESULTS_DIR}/ppg_ast_pd_v3_cv_results.npz')
print(f'Saved: {RESULTS_DIR}/ppg_ast_pd_v3_fold{{1-5}}.pt')

In [ ]:
#9 - PPG vs Spectrogram comparison
print('='*70)
print('PPG-AST vs Spectrogram-AST Comparison')
print('='*70)

# PPG results (just computed)
print('\n--- PPG-AST (this experiment) ---')
print(f'  OOF AUC:       {oof_auc:.4f}')
print(f'  OOF F1:        {oof_f1:.4f}')
print(f'  OOF Recall:    {oof_rec:.4f}')
print(f'  OOF Precision: {oof_prec:.4f}')
print(f'  OOF Accuracy:  {oof_acc:.4f}')
print(f'  Mean CV AUC:   {np.mean(aucs):.4f} (+/- {np.std(aucs, ddof=1):.4f})')
print(f'  Mean CV F1:    {np.mean(f1s):.4f} (+/- {np.std(f1s, ddof=1):.4f})')

# Spectrogram results (if available)
if SPEC_RESULTS.exists():
    spec_res = np.load(str(SPEC_RESULTS), allow_pickle=True)
    spec_oof_probs = spec_res['oof_probs']
    spec_oof_labels = spec_res['oof_labels']
    spec_fold_aucs = spec_res['fold_aucs']
    spec_fold_f1s = spec_res['fold_f1s']

    spec_oof_auc = roc_auc_score(spec_oof_labels, spec_oof_probs)
    fpr_s, tpr_s, thr_s = roc_curve(spec_oof_labels, spec_oof_probs)
    opt_s = np.argmax(tpr_s - fpr_s)
    spec_preds = (spec_oof_probs >= thr_s[opt_s]).astype(int)
    spec_oof_f1 = f1_score(spec_oof_labels, spec_preds, zero_division=0)
    spec_oof_rec = recall_score(spec_oof_labels, spec_preds, zero_division=0)
    spec_oof_prec = precision_score(spec_oof_labels, spec_preds, zero_division=0)
    spec_oof_acc = accuracy_score(spec_oof_labels, spec_preds)

    print('\n--- Spectrogram-AST (from saved results) ---')
    print(f'  OOF AUC:       {spec_oof_auc:.4f}')
    print(f'  OOF F1:        {spec_oof_f1:.4f}')
    print(f'  OOF Recall:    {spec_oof_rec:.4f}')
    print(f'  OOF Precision: {spec_oof_prec:.4f}')
    print(f'  OOF Accuracy:  {spec_oof_acc:.4f}')
    print(f'  Mean CV AUC:   {np.mean(spec_fold_aucs):.4f} (+/- {np.std(spec_fold_aucs, ddof=1):.4f})')
    print(f'  Mean CV F1:    {np.mean(spec_fold_f1s):.4f} (+/- {np.std(spec_fold_f1s, ddof=1):.4f})')

    print('\n--- Head-to-head ---')
    print(f'  {"Metric":<18} {"PPG-AST":>10} {"Spec-AST":>10} {"Delta":>10}')
    print(f'  {"-"*48}')
    for name, ppg_val, spec_val in [
        ('OOF AUC', oof_auc, spec_oof_auc),
        ('OOF F1', oof_f1, spec_oof_f1),
        ('OOF Recall', oof_rec, spec_oof_rec),
        ('OOF Precision', oof_prec, spec_oof_prec),
        ('OOF Accuracy', oof_acc, spec_oof_acc),
        ('Mean CV AUC', np.mean(aucs), np.mean(spec_fold_aucs)),
        ('Mean CV F1', np.mean(f1s), np.mean(spec_fold_f1s)),
    ]:
        delta = ppg_val - spec_val
        sign = '+' if delta >= 0 else ''
        print(f'  {name:<18} {ppg_val:>10.4f} {spec_val:>10.4f} {sign}{delta:>9.4f}')
else:
    print('\n--- Spectrogram results not found ---')
    print(f'  Expected at: {SPEC_RESULTS}')
    print('  Run v3_PD_AST_Spectrograms.ipynb first to enable comparison.')